# 01 · Chennai GTFS data audit

**Purpose:** load the supplied static feed, check its structure and key relationships, and record the feed's limits before analysis.

This feed contains scheduled services, routes, stops, and timings. It does **not** contain observed passenger boardings, alightings, or route-level ridership. Do not treat a scheduled trip or stop as a passenger count.

Run from the project root with the `chennai-gtfs` environment activated. See the README for setup and the `GTFS_DIR` option.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir, load_gtfs

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
feed = load_gtfs(GTFS_DIR)
print(f"Feed folder: {GTFS_DIR}")


## 1. Table sizes and columns

In [ ]:
for name, df in feed.items():
    print(f"{name:12s} rows={len(df):,} columns={len(df.columns)}")
    display(df.head(3))

## 2. Required fields, missing values, and key uniqueness

In [ ]:
key_cols = {"agency": ["agency_id"], "calendar": ["service_id"], "routes": ["route_id"],
            "stops": ["stop_id"], "trips": ["trip_id"]}
for name, cols in key_cols.items():
    df = feed[name]
    missing_cols = sorted(set(cols) - set(df.columns))
    duplicate_keys = int(df.duplicated(cols).sum()) if not missing_cols else None
    print(f"{name}: missing key columns={missing_cols}; duplicate keys={duplicate_keys}")

for name, df in feed.items():
    blank_cells = int(df.replace("", pd.NA).isna().sum().sum())
    print(f"{name}: blank cells={blank_cells:,}")

## 3. Foreign-key checks and service date coverage

In [ ]:
route_ids = set(feed["routes"]["route_id"])
trip_route_ids = set(feed["trips"]["route_id"])
trip_ids = set(feed["trips"]["trip_id"])
stop_ids = set(feed["stops"]["stop_id"])
service_ids = set(feed["calendar"]["service_id"])

checks = {
    "trip route IDs absent from routes.txt": len(trip_route_ids - route_ids),
    "trip service IDs absent from calendar.txt": len(set(feed["trips"]["service_id"]) - service_ids),
    "stop_times trip IDs absent from trips.txt": len(set(feed["stop_times"]["trip_id"]) - trip_ids),
    "stop_times stop IDs absent from stops.txt": len(set(feed["stop_times"]["stop_id"]) - stop_ids),
}
for label, count in checks.items():
    print(f"{label}: {count:,}")

cal = feed["calendar"]
print("Calendar date span:", cal["start_date"].min().date(), "to", cal["end_date"].max().date())
print("Agency names:", feed["agency"]["agency_name"].unique().tolist())
print("Route type counts:")
display(feed["routes"]["route_type"].value_counts(dropna=False).rename_axis("route_type").to_frame("routes"))

## 4. What this audit tells us

- `trips.txt` and `stop_times.txt` measure **scheduled supply**, not passengers.
- `calendar.txt` says when a service is scheduled to operate. The feed also has no `calendar_dates.txt`, so this first version cannot model service exceptions or cancellations.
- For actual route-level ridership, seek MTC ticketing / passenger-count data or collect a small survey. If the project must proceed without it, notebook 03 creates a **synthetic scenario** calibrated to an aggregate daily total. That is a planning assumption, not ground truth.

Save the audit findings in your report, including any nonzero key-check results.